# MAE Only Training Notebook
Train MAE up to 120 epochs with Warmup + Cosine scheduler, early stopping, visual sanity checks, and checkpointing.

In [ ]:
# --- Section 0: Setup Project Environment (Kaggle) ---
import os
import sys
import shutil
import subprocess

pkgs = [
    "transformers>=4.30",
    "matplotlib>=3.7",
    "pillow>=10.0",
    "numpy>=1.24",
    "scikit-learn>=1.3",
]

print("Installing extra dependencies...")
cmd = [sys.executable, "-m", "pip", "install", "-q", "--no-input", *pkgs]
subprocess.run(cmd, check=True)
print("Install complete\n")

SOURCE_DIR = "/kaggle/input/datasets/priyadatechasaratool/mae-classification-code01"
WORK_DIR = "/kaggle/working/project"

if not os.path.exists(SOURCE_DIR):
    raise FileNotFoundError(
        f"SOURCE_DIR not found: {SOURCE_DIR}. Please update SOURCE_DIR."
    )

if not os.path.exists(WORK_DIR):
    print(f"Copying files from\n  {SOURCE_DIR}\n-> {WORK_DIR} ...")
    shutil.copytree(SOURCE_DIR, WORK_DIR)
    print("Copy complete!\n")
else:
    print(f"Already exists: {WORK_DIR}\n")

os.chdir(WORK_DIR)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

print(f"CWD: {os.getcwd()}")

In [ ]:
# --- Section 1: Config and Runtime ---
from dataclasses import dataclass
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch

from utils.common import set_seed


@dataclass
class Config:
    DATA_ROOT: str = "/kaggle/input/datasets/alessiocorrado99/animals10/raw-img"
    CKPT_DIR: str = "/kaggle/working/checkpoints_mae_only"
    RESULT_DIR: str = "/kaggle/working/results_mae_only"
    BATCH_SIZE: int = 32
    EPOCHS_MAE: int = 120
    LR: float = 1e-4
    MASK_RATIO: float = 0.75
    IMG_SIZE: int = 224
    SEED: int = 42
    NUM_WORKERS: int = 2
    USE_WEIGHTED_SAMPLER: bool = True
    EARLY_STOPPING_PATIENCE: int = 20
    MIN_EPOCHS_BEFORE_EARLY_STOP: int = 30 
    EARLY_STOPPING_MIN_DELTA: float = 2e-5
    WARMUP_RATIO: float = 0.03
    VIS_EVERY_EPOCHS: int = 5
    CKPT_EVERY_EPOCHS: int = 5


cfg = Config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    try:
        _ = torch.zeros(1, device=device) + 1
        torch.cuda.synchronize()
    except Exception as exc:
        print(f"CUDA unavailable at runtime ({exc}). Fallback to CPU.")
        device = torch.device("cpu")

set_seed(cfg.SEED)

Path(cfg.CKPT_DIR).mkdir(parents=True, exist_ok=True)
Path(cfg.RESULT_DIR).mkdir(parents=True, exist_ok=True)

print(f"Device: {device}")
print(f"Dataset root: {cfg.DATA_ROOT}")
print(f"Checkpoint dir: {cfg.CKPT_DIR}")
print(f"Result dir: {cfg.RESULT_DIR}")
print(f"Warmup ratio: {cfg.WARMUP_RATIO}")
print(f"Visualize every N epochs: {cfg.VIS_EVERY_EPOCHS}")
print(f"Checkpoint every N epochs: {cfg.CKPT_EVERY_EPOCHS}")

In [ ]:
# --- Section 2: Load and Inspect Dataset ---
from collections import Counter
from PIL import Image

from data.animals10 import (
    build_split,
    discover_class_directories,
    map_animals10_labels_to_english,
)

class_dirs = discover_class_directories(cfg.DATA_ROOT)
split = build_split(cfg.DATA_ROOT, val_fraction=0.2, seed=cfg.SEED)
class_to_idx_en, idx_to_class_en = map_animals10_labels_to_english(split.class_to_idx)

print(f"Classes found: {len(class_dirs)}")
print("Original folder names (IT):", sorted(split.class_to_idx.keys()))
print("Classification labels (EN):", [idx_to_class_en[i] for i in sorted(idx_to_class_en.keys())])
print(f"Train samples: {len(split.train_samples)}")
print(f"Val samples: {len(split.val_samples)}")

train_counts = Counter(label for _, label in split.train_samples)
print("Train distribution:", {idx_to_class_en[idx]: count for idx, count in train_counts.items()})

random_samples = random.sample(split.train_samples, 4)
sample_paths = [path for path, _ in random_samples]

fig, axes = plt.subplots(1, len(sample_paths), figsize=(16, 4))
for axis, sample_path in zip(axes, sample_paths):
    with Image.open(sample_path) as image:
        axis.imshow(image.convert("RGB"))
    axis.set_title(Path(sample_path).parent.name)
    axis.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# --- Section 3: Transforms and DataLoaders ---
from torchvision import transforms as T
from data.animals10 import IMAGENET_MEAN, IMAGENET_STD, build_dataloaders

train_transform = T.Compose(
    [
        T.RandomResizedCrop(
            cfg.IMG_SIZE,
            scale=(0.2, 1.0),
            interpolation=T.InterpolationMode.BICUBIC,
        ),
        T.RandomHorizontalFlip(p=0.5),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)

val_transform = T.Compose(
    [
        T.Resize(256, interpolation=T.InterpolationMode.BICUBIC),
        T.CenterCrop(cfg.IMG_SIZE),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)

train_loader, val_loader, split = build_dataloaders(
    root=cfg.DATA_ROOT,
    batch_size=cfg.BATCH_SIZE,
    val_fraction=0.2,
    seed=cfg.SEED,
    num_workers=cfg.NUM_WORKERS,
    train_transform=train_transform,
    val_transform=val_transform,
    use_weighted_sampler=cfg.USE_WEIGHTED_SAMPLER,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

In [ ]:
# --- Section 4: Build MAE and Training Utilities ---
from transformers import get_cosine_schedule_with_warmup

from training.mae_trainer import MAETrainer, load_mae_model, reconstruct_mae_images
from utils.common import EarlyStopping, create_grad_scaler, save_json, tensor_to_numpy_image

mae_model = load_mae_model(mask_ratio=cfg.MASK_RATIO).to(device)
mae_optimizer = torch.optim.AdamW(mae_model.parameters(), lr=cfg.LR)

total_steps = len(train_loader) * cfg.EPOCHS_MAE
warmup_steps = max(1, int(total_steps * cfg.WARMUP_RATIO))
mae_scheduler = get_cosine_schedule_with_warmup(
    optimizer=mae_optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
 )

mae_scaler = create_grad_scaler(device)
mae_trainer = MAETrainer(
    mae_model,
    mae_optimizer,
    device,
    mask_ratio=cfg.MASK_RATIO,
    scaler=mae_scaler,
)
mae_early_stopper = EarlyStopping(
    patience=cfg.EARLY_STOPPING_PATIENCE,
    min_delta=cfg.EARLY_STOPPING_MIN_DELTA,
    mode="min",
)

print(f"Total steps: {total_steps}")
print(f"Warmup steps: {warmup_steps}")

In [ ]:
# --- Section 5: Train MAE (120 epochs max) ---
from training.unet import apply_patch_mask

best_mae_val = float("inf")
history = []
best_epoch = 0
visual_dir = Path(cfg.RESULT_DIR) / "visual_checks"
visual_dir.mkdir(parents=True, exist_ok=True)

print(f"--- Starting MAE Training ({cfg.EPOCHS_MAE} Epochs max) ---")
for epoch in range(cfg.EPOCHS_MAE):
    mae_train_loss = mae_trainer.train_epoch(train_loader, scheduler=mae_scheduler)
    mae_val_loss = mae_trainer.evaluate_epoch(val_loader)

    current_lr = mae_optimizer.param_groups[0]["lr"]
    epoch_metrics = {
        "epoch": epoch + 1,
        "train_loss": float(mae_train_loss),
        "val_loss": float(mae_val_loss),
        "lr": float(current_lr),
    }
    history.append(epoch_metrics)

    # Save periodic checkpoints every N epochs
    if (epoch + 1) % cfg.CKPT_EVERY_EPOCHS == 0 or (epoch + 1) == cfg.EPOCHS_MAE:
        mae_trainer.save_checkpoint(
            Path(cfg.CKPT_DIR) / f"mae_epoch_{epoch+1:03d}.pt",
            epoch=epoch + 1,
            metrics=epoch_metrics,
            config={"mask_ratio": cfg.MASK_RATIO},
        )

    if mae_val_loss < best_mae_val:
        best_mae_val = mae_val_loss
        best_epoch = epoch + 1
        mae_trainer.save_checkpoint(
            Path(cfg.CKPT_DIR) / "mae_best.pt",
            epoch=epoch + 1,
            metrics=epoch_metrics,
            config={"mask_ratio": cfg.MASK_RATIO},
        )

    print(
        f"MAE Epoch {epoch+1}/{cfg.EPOCHS_MAE} | "
        f"Train Loss: {mae_train_loss:.4f} | Val Loss: {mae_val_loss:.4f} | "
        f"LR: {current_lr:.2e}"
    )

    # Visual sanity check every N epochs and on final epoch
    if (epoch + 1) % cfg.VIS_EVERY_EPOCHS == 0 or (epoch + 1) == cfg.EPOCHS_MAE:
        print(f"Visualizing reconstructions for epoch {epoch+1}...")
        val_batch = next(iter(val_loader))
        sample_images = val_batch[0][:4].to(device)
        masked_images, _ = apply_patch_mask(sample_images, mask_ratio=cfg.MASK_RATIO)

        mae_model.eval()
        with torch.no_grad():
            recon_images = reconstruct_mae_images(mae_model, sample_images)

        num_images = sample_images.size(0)
        fig, axes = plt.subplots(3, num_images, figsize=(4 * num_images, 9))
        fig.suptitle(f"Epoch {epoch+1} Reconstructions", fontsize=14)

        for i in range(num_images):
            axes[0, i].imshow(tensor_to_numpy_image(sample_images[i]))
            axes[0, i].set_title("Original")
            axes[0, i].axis("off")

            axes[1, i].imshow(tensor_to_numpy_image(masked_images[i]))
            axes[1, i].set_title("Masked")
            axes[1, i].axis("off")

            axes[2, i].imshow(tensor_to_numpy_image(recon_images[i]))
            axes[2, i].set_title("Reconstructed")
            axes[2, i].axis("off")

        visual_path = visual_dir / f"epoch_{epoch+1:03d}.png"
        plt.tight_layout()
        plt.savefig(visual_path, dpi=160, bbox_inches="tight")
        plt.show()
        plt.close()

        mae_model.train()

    if (epoch + 1) >= cfg.EARLY_STOPPING_START_EPOCH and mae_early_stopper.step(mae_val_loss):
        print(f"Early stopping MAE at epoch {epoch+1}")
        break

save_json(Path(cfg.RESULT_DIR) / "mae_train_history.json", history)
save_json(
    Path(cfg.RESULT_DIR) / "mae_summary.json",
    {
        "best_val_loss": float(best_mae_val),
        "best_epoch": int(best_epoch),
        "last_epoch": int(history[-1]["epoch"]) if history else 0,
        "best_checkpoint": str(Path(cfg.CKPT_DIR) / "mae_best.pt"),
        "visual_check_dir": str(visual_dir),
    },
)

print("MAE training complete.")
print("Saved best checkpoint (recommended for classifier init):", Path(cfg.CKPT_DIR) / "mae_best.pt")